# Étape 2 — Suivi des expérimentations avec MLflow

L’objectif de cette étape est de tracer les expérimentations de modélisation avec MLflow.

Pour chaque modèle, les paramètres, les métriques et le modèle entraîné seront enregistrés afin de comparer les performances dans l’interface MLflow.

Les métriques suivies sont :
- accuracy ;
- precision ;
- recall ;
- F1-score ;
- ROC AUC ;
- matrice de confusion ;
- score métier basé sur le coût des erreurs.

Dans ce projet, les faux négatifs sont particulièrement coûteux : un faux négatif correspond à un client risqué prédit comme non risqué, donc à un crédit potentiellement accordé à un client susceptible de ne pas rembourser.

Les expérimentations de modélisation ont été tracées avec MLflow.

Chaque modèle est entraîné dans un script Python reproductible situé dans `src/models/`. Les scripts utilisent `mlflow.start_run()` pour créer un run MLflow et enregistrer les paramètres, les métriques, les tags et le modèle entraîné.

Les modèles suivis sont :

- `DummyClassifier`, utilisé comme baseline naïve ;

- `LogisticRegression`, avec imputation, scaling et pondération des classes ;

- `LightGBM`, avec gestion du déséquilibre via `scale_pos_weight` ;

- `LightGBM` avec optimisation du seuil de décision.

L’interface MLflow permet ensuite de comparer les runs selon les métriques classiques et selon un score métier.

In [ ]:
import mlflow
import pandas as pd

EXPERIMENT_NAME = "P6_credit_scoring_baseline"

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
)

columns_to_display = [
    "tags.mlflow.runName",
    "params.model",
    "params.threshold",
    "metrics.accuracy",
    "metrics.precision",
    "metrics.recall",
    "metrics.f1_score",
    "metrics.roc_auc",
    "metrics.fn",
    "metrics.fp",
    "metrics.business_cost",
    "metrics.business_cost_per_client",
    "metrics.best_threshold",
    "metrics.best_business_cost",
]

available_columns = [
    column for column in columns_to_display
    if column in runs.columns
]

runs[available_columns].sort_values(
    by=[column for column in ["metrics.business_cost", "metrics.best_business_cost"] if column in runs.columns],
    ascending=True,
)

## Baseline — DummyClassifier

Un premier modèle baseline a été entraîné avec `DummyClassifier(strategy="most_frequent")`.

Ce modèle prédit systématiquement la classe majoritaire, c’est-à-dire `TARGET = 0`, qui correspond aux clients sans défaut.

Résultats obtenus sur le jeu de validation :

| Métrique | Valeur |
|---|---:|
| Accuracy | 0.9192 |
| Precision | 0.0000 |
| Recall | 0.0000 |
| F1-score | 0.0000 |
| ROC AUC | 0.5000 |
| Faux négatifs | 4965 |
| Faux positifs | 0 |
| Coût métier | 49650 |

L’accuracy est élevée car le dataset est fortement déséquilibré : environ 92 % des clients ne sont pas en défaut. Cependant, le modèle ne détecte aucun client risqué. Le recall est donc nul.

Ce baseline montre que l’accuracy seule n’est pas suffisante pour évaluer un modèle de scoring crédit. Il faudra privilégier des métriques comme le recall, l’AUC et le score métier.

Le premier modèle baseline est un DummyClassifier qui prédit systématiquement la classe majoritaire, c’est-à-dire TARGET = 0.

Il obtient une accuracy élevée d’environ 91,9 %, car la majorité des clients du dataset ne sont pas en défaut. Cependant, cette métrique est trompeuse dans un contexte de classes déséquilibrées.

L’accuracy mesure uniquement la proportion globale de bonnes prédictions. Elle ne tient pas compte de la classe que le modèle détecte réellement, ni du coût des erreurs. Dans ce cas, le modèle prédit tous les clients comme non risqués : il obtient donc beaucoup de vrais négatifs, mais il ne détecte aucun client en défaut.

La matrice de confusion confirme ce problème : le modèle produit 4965 faux négatifs et 0 vrai positif. Cela signifie que tous les clients réellement en défaut sont classés comme bons clients.

Dans un projet de scoring crédit, cette erreur est particulièrement grave, car un faux négatif correspond à un crédit potentiellement accordé à un client risqué. C’est pourquoi l’accuracy seule est insuffisante. Il faut aussi suivre le recall, la précision, le F1-score, l’AUC et un score métier qui pénalise davantage les faux négatifs.

## Modèle 2 — LogisticRegression avec class_weight balanced

Un second modèle a été entraîné avec une `LogisticRegression` intégrée dans un pipeline scikit-learn. Le pipeline contient une imputation par médiane, un scaling avec `StandardScaler`, puis le modèle de régression logistique.

Le paramètre `class_weight="balanced"` permet de tenir compte du déséquilibre des classes en donnant plus de poids à la classe minoritaire `TARGET = 1`.

Par rapport au `DummyClassifier`, l’accuracy diminue, passant d’environ 91,9 % à 71,0 %. Cette baisse est normale, car le modèle ne prédit plus systématiquement la classe majoritaire. En revanche, le recall passe de 0 à environ 70,5 %, ce qui signifie que le modèle détecte une grande partie des clients réellement en défaut.

La matrice de confusion montre que le nombre de faux négatifs passe de 4965 à 1465. Le modèle détecte donc 3500 clients en défaut qui étaient totalement ignorés par le baseline.

Le coût métier diminue également, passant de 49650 à 31024. Cela montre que, même avec une accuracy plus faible, la régression logistique est plus pertinente métier que le modèle dummy.

Les expérimentations sont suivies dans MLflow au sein de l’expérience `P6_credit_scoring_baseline`. Chaque run enregistre les paramètres du modèle, les métriques classiques, la matrice de confusion, le score métier ainsi que des tags permettant d’identifier le type de modèle et la version du code utilisée.

La comparaison MLflow montre que le DummyClassifier obtient une accuracy élevée, mais uniquement parce qu’il prédit systématiquement la classe majoritaire. Il ne détecte aucun client en défaut, avec un recall égal à 0 et 4965 faux négatifs.

La LogisticRegression obtient une accuracy plus faible, mais elle détecte une partie importante des clients en défaut. Son recall atteint environ 70,5 %, son AUC atteint environ 0,773 et le nombre de faux négatifs diminue de 4965 à 1465.

D’un point de vue métier, la LogisticRegression est donc plus pertinente que le DummyClassifier, car elle réduit le coût métier de 49650 à 31024.

## Modèle 3 — LightGBM baseline pondéré

Un troisième modèle a été entraîné avec LightGBM. Ce modèle est adapté aux données tabulaires et gère nativement les valeurs manquantes. Contrairement à la régression logistique, il ne nécessite pas de scaling.

Le déséquilibre de classes a été pris en compte avec le paramètre `scale_pos_weight`, calculé à partir du ratio entre les clients non-défaillants et les clients en défaut dans le jeu d’entraînement.

Le modèle obtient une accuracy d’environ 73,8 %, un recall d’environ 69,2 % et une AUC d’environ 0,788. Il détecte 3434 clients en défaut et en rate 1531.

Par rapport à la LogisticRegression, LightGBM obtient une AUC plus élevée et un coût métier plus faible. Le coût métier passe de 31024 pour la LogisticRegression à 29912 pour LightGBM. À ce stade, LightGBM est donc le meilleur modèle provisoire selon le critère métier.

| Modèle | Accuracy | Recall | AUC | FN | FP | Coût métier |
|---|---:|---:|---:|---:|---:|---:|
| DummyClassifier | 0.9193 | 0.0000 | 0.5000 | 4965 | 0 | 49650 |
| LogisticRegression | 0.7099 | 0.7049 | 0.7732 | 1465 | 16374 | 31024 |
| LightGBM | 0.7377 | 0.6916 | 0.7881 | 1531 | 14602 | 29912 |

## Optimisation du seuil de décision

Après l'entraînement du modèle LightGBM, plusieurs seuils de décision ont été testés afin de minimiser le coût métier.

Le seuil par défaut de 0.5 donne un coût métier de 29912, avec 1531 faux négatifs et 14602 faux positifs.

L'optimisation du seuil indique qu'un seuil de 0.55 minimise légèrement le coût métier, avec un coût de 29890. Ce seuil réduit le nombre de faux positifs à 11720, mais augmente le nombre de faux négatifs à 1817. Le recall diminue donc de 0.692 à 0.634.

Le gain métier reste limité, mais cette étape montre l'intérêt de ne pas se limiter au seuil par défaut de 0.5. Dans un problème de scoring crédit, le choix du seuil doit être guidé par le coût métier des erreurs, et non uniquement par l'accuracy.

## Versionnement du meilleur modèle dans MLflow

Après comparaison des différents runs MLflow, le modèle LightGBM avec optimisation du seuil obtient le meilleur coût métier.

Le meilleur seuil identifié est 0.55, avec un coût métier de 29890. Ce modèle a été enregistré dans le Model Registry sous le nom `P6_credit_scoring_lightgbm_champion`.

Une première version du modèle a été créée et l’alias `champion` lui a été attribué. Cela permet d’identifier clairement le meilleur modèle actuel et de préparer son utilisation dans les prochaines étapes du cycle MLOps.